# LeetCode 1430 — Check If a String Is a Valid Sequence from Root to Leaves Path in a Binary Tree bằng Python

Link đề: https://leetcode.com/problems/check-if-a-string-is-a-valid-sequence-from-root-to-leaves-path-in-a-binary-tree/ · Độ khó: **Medium** · Chủ đề: Tree, Depth-First Search, Breadth-First Search, Binary Tree

> **Ghi chú Premium:** đây là bài **LeetCode Premium** (🔒). Đề bài và chữ ký trong notebook được viết theo hiểu biết về bài, lấy từ trang lời giải doocs (https://leetcode.doocs.org/en/lc/1430/). Nếu có tài khoản Premium, nên đối chiếu lại với đề gốc.

Skeleton LeetCode (theo doocs):

```python
# Definition for a binary tree node.
# class TreeNode:
#     def __init__(self, val=0, left=None, right=None):
#         self.val = val
#         self.left = left
#         self.right = right
class Solution:
    def isValidSequence(self, root: TreeNode, arr: list[int]) -> bool:
        ...
```

## 1. Mục tiêu học

1. Hiểu khái niệm **đường đi từ gốc tới lá** (root-to-leaf path) và điều kiện **"phải kết thúc ở lá"**.
2. Viết **DFS mang theo chỉ số** `u` trong mảng: so `root.val` với `arr[u]`, cắt tỉa ngay khi lệch.
3. Viết **BFS theo tầng**: tầng `u` chỉ giữ các nút khớp với `arr[u]` — "tập biên" thu hẹp dần.
4. Cài **DFS lặp bằng stack** để tránh giới hạn đệ quy khi cây suy biến sâu (tới 5000 tầng).
5. Tránh các lỗi: quên kiểm tra lá, kiểm tra lá sai (`or` thay vì `and`), đặt base case ở nút `None`, chỉ đi một nhánh.

## 2. Đề bài & ví dụ

Cho một cây nhị phân, mỗi đường đi từ **gốc tới một lá** tạo thành một **dãy hợp lệ** (valid sequence) — dãy các giá trị nút theo thứ tự trên đường đi. Cho mảng số nguyên `arr`, kiểm tra xem `arr` có phải là **một dãy hợp lệ** của cây hay không, tức là tồn tại đường đi gốc → lá mà dãy giá trị nút **bằng đúng** `arr`.

(Tên đề nói "string" vì có thể coi `arr` là chuỗi ghép từ các chữ số.)

Cây dùng trong các ví dụ: `root = [0,1,0,0,1,0,null,null,1,0,0]`

```
            0
          /   \
         1     0
        / \   /
       0   1 0
        \  /\
        1 0  0
```

**Ví dụ 1:** `arr = [0,1,0,1]` → `true` (đường `0 → 1 → 0 → 1`). Các dãy hợp lệ khác: `0 → 1 → 1 → 0`, `0 → 0 → 0`.

**Ví dụ 2:** `arr = [0,0,1]` → `false` (đường `0 → 0 → 1` không tồn tại).

**Ví dụ 3:** `arr = [0,1,1]` → `false` (đường `0 → 1 → 1` có tồn tại nhưng **không kết thúc ở lá**).

**Ràng buộc:** `1 <= arr.length <= 5000`, `0 <= arr[i] <= 9`, giá trị mỗi nút trong `[0, 9]`.

In [1]:
import random
import sys
import time
from collections import deque
from typing import List, Optional

class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right

def build_tree(level):
    """Dựng cây từ mảng level-order kiểu LeetCode (None = không có nút)."""
    if not level or level[0] is None:
        return None
    root = TreeNode(level[0])
    q = deque([root])
    i = 1
    while q and i < len(level):
        node = q.popleft()
        if i < len(level) and level[i] is not None:
            node.left = TreeNode(level[i]); q.append(node.left)
        i += 1
        if i < len(level) and level[i] is not None:
            node.right = TreeNode(level[i]); q.append(node.right)
        i += 1
    return root

def to_level(root):
    """Ngược lại: cây -> mảng level-order (bỏ None thừa ở cuối)."""
    out, q = [], deque([root])
    while q:
        node = q.popleft()
        if node is None:
            out.append(None); continue
        out.append(node.val); q.append(node.left); q.append(node.right)
    while out and out[-1] is None:
        out.pop()
    return out

T = [0, 1, 0, 0, 1, 0, None, None, 1, 0, 0]
root = build_tree(T)
print(to_level(root) == T)
examples = [([0, 1, 0, 1], True), ([0, 0, 1], False), ([0, 1, 1], False)]
for arr, want in examples:
    print(arr, "->", want)

True
[0, 1, 0, 1] -> True
[0, 0, 1] -> False
[0, 1, 1] -> False


## 3. Kiến thức nền

### 3.1 Cây nhị phân, nút lá

Mỗi `TreeNode` có `val`, `left`, `right`. **Lá** là nút **không có con nào**: `node.left is None and node.right is None`. Nút chỉ có một con **không phải** lá.

In [2]:
def is_leaf(node):
    return node is not None and node.left is None and node.right is None

print("gốc là lá?", is_leaf(root))
print("nút root.right.left là lá?", is_leaf(root.right.left))
print("nút root.left.left (có con phải) là lá?", is_leaf(root.left.left))

gốc là lá? False
nút root.right.left là lá? True
nút root.left.left (có con phải) là lá? False


### 3.2 In cây và liệt kê đường đi gốc → lá

Duyệt đệ quy, mang theo `path`; tới lá thì ghi lại bản sao `path`.

In [3]:
def all_paths(root):
    res, path = [], []
    def dfs(node):
        if node is None:
            return
        path.append(node.val)
        if node.left is None and node.right is None:
            res.append(path[:])
        else:
            dfs(node.left); dfs(node.right)
        path.pop()
    dfs(root)
    return res

for p in all_paths(root):
    print(" -> ".join(map(str, p)))

0 -> 1 -> 0 -> 1
0 -> 1 -> 1 -> 0
0 -> 1 -> 1 -> 0
0 -> 0 -> 0


### 3.3 Đệ quy với chỉ số và toán tử `or` ngắn mạch

`dfs(left, u+1) or dfs(right, u+1)`: nếu nhánh trái đã `True` thì **không** gọi nhánh phải (short-circuit) ⇒ dừng sớm.

In [4]:
def noisy(name, val):
    print("  gọi", name)
    return val

print(noisy("trái", True) or noisy("phải", True))
print(noisy("trái", False) or noisy("phải", True))

  gọi trái
True
  gọi trái
  gọi phải
True


### 3.4 Giới hạn đệ quy của Python

Mặc định khoảng 1000 tầng. Cây suy biến (mỗi nút một con) với `arr` dài 5000 có thể sâu 5000 ⇒ đệ quy có thể lỗi `RecursionError`. Cách xử lý: `sys.setrecursionlimit` hoặc viết **lặp bằng stack / BFS**.

In [5]:
print("giới hạn đệ quy hiện tại:", sys.getrecursionlimit())

giới hạn đệ quy hiện tại: 1000


### 3.5 BFS theo tầng bằng `deque` / list

Tầng `u` của cây = các nút ở độ sâu `u`. Ta có thể giữ một **list các nút ở tầng hiện tại** rồi dựng list tầng kế tiếp.

In [6]:
level, depth = [root], 0
while level:
    print("tầng", depth, [n.val for n in level])
    level = [c for n in level for c in (n.left, n.right) if c]
    depth += 1

tầng 0 [0]
tầng 1 [1, 0]
tầng 2 [0, 1, 0]
tầng 3 [1, 0, 0]


## 4. Ý tưởng nền tảng / trực giác

Kiểm tra tay `arr = [0, 1, 0, 1]`:
1. `arr[0] = 0`: gốc là 0 ✔.
2. `arr[1] = 1`: con trái của gốc là 1 ✔ (con phải là 0 ✘ — bỏ nhánh này).
3. `arr[2] = 0`: con trái của nút 1 là 0 ✔ (con phải 1 ✘).
4. `arr[3] = 1`: nút 0 đó chỉ có con phải là 1 ✔. Đây là **phần tử cuối** của `arr` — nút 1 này có phải **lá** không? Có ⇒ `true`.

Với `arr = [0, 1, 1]`: tới `arr[2] = 1` ta ở nút `1` (con phải của nút 1), đây là phần tử cuối nhưng nút này **có hai con** ⇒ không phải lá ⇒ nhánh này thất bại; không nhánh nào khác khớp ⇒ `false`.

Quy tắc: ở mỗi bước, **lệch giá trị** ⇒ dừng nhánh; **hết `arr`** ⇒ chỉ đúng khi đang ở **lá**; **hết cây** (gặp `None`) trước khi hết `arr` ⇒ sai.

In [7]:
arr = [0, 1, 0, 1]
node = root
print("bước 0: gốc", node.val, "== arr[0]?", node.val == arr[0])
for u in range(1, len(arr)):
    cand = [c for c in (node.left, node.right) if c and c.val == arr[u]]
    print(f"bước {u}: các con khớp arr[{u}]={arr[u]}: {[c.val for c in cand]}")
    node = cand[0]
print("nút cuối là lá?", node.left is None and node.right is None)

bước 0: gốc 0 == arr[0]? True
bước 1: các con khớp arr[1]=1: [1]
bước 2: các con khớp arr[2]=0: [0]
bước 3: các con khớp arr[3]=1: [1]
nút cuối là lá? True


## 5. Cách 1 — Brute Force / baseline: liệt kê mọi đường đi gốc → lá

**Ý tưởng:** dùng `all_paths` sinh **tất cả** đường đi gốc → lá, rồi kiểm tra `arr` có nằm trong đó không. Đúng ngay theo định nghĩa.

In [8]:
def valid_brute(root, arr):
    return any(p == arr for p in all_paths(root))

for arr, want in examples:
    print(arr, valid_brute(root, arr), "✔" if valid_brute(root, arr) == want else "✘")

[0, 1, 0, 1] True ✔
[0, 0, 1] False ✔
[0, 1, 1] False ✔


**Độ phức tạp:** gọi `N` = số nút, `H` = chiều cao. Số lá ≤ `N`, mỗi đường đi dài ≤ `H` ⇒ sao chép tốn `O(N · H)` thời gian và bộ nhớ (cây cân bằng `O(N log N)`, cây "chổi" có thể `O(N²)`).

**Nút thắt:** sinh cả những đường đi lệch `arr` ngay từ nút đầu tiên. Nên **so sánh trong lúc đi** và **cắt tỉa** nhánh lệch.

In [9]:
def broom(depth, leaves_per_level=1):
    """Cây 'chổi': một sống dài, mỗi nút trên sống có thêm một lá."""
    root = TreeNode(0); cur = root
    for _ in range(depth):
        cur.left = TreeNode(1)       # lá
        cur.right = TreeNode(0)      # tiếp tục sống
        cur = cur.right
    return root

b = broom(1500)
sys.setrecursionlimit(10000)
t = time.perf_counter(); valid_brute(b, [0] * 1501); el = time.perf_counter() - t
print(f"brute trên cây chổi 1500 tầng: {el:.3f}s, tổng độ dài các đường ≈ {sum(len(p) for p in all_paths(b))}")

brute trên cây chổi 1500 tầng: 0.008s, tổng độ dài các đường ≈ 1128751


## 6. Ý tưởng thuật toán / bước đột phá

**Insight 1 — so khớp từng tầng:** nút ở độ sâu `u` chỉ có thể khớp với `arr[u]`. Mang chỉ số `u` theo khi đi xuống.

**Insight 2 — cắt tỉa:** **nếu** `node is None` hoặc `node.val != arr[u]` **thì** nhánh này thất bại ngay.

**Insight 3 — điều kiện kết thúc ở phần tử cuối:** **nếu** `u == len(arr) − 1` (đã khớp phần tử cuối) **thì** kết quả là `node` **là lá** — không đi tiếp.

**Insight 4 — BFS tương đương:** tập `F_u` = các nút ở độ sâu `u` mà đường từ gốc tới nó khớp `arr[0..u]`. `F_0 = {root}` nếu khớp; `F_{u+1}` = các con của nút trong `F_u` có giá trị `arr[u+1]`. Đáp án: có nút lá trong `F_{len−1}`.

Số nút được thăm ≤ `min(N, ...)` và mỗi nút thăm `O(1)` ⇒ `O(N)` thời gian; thường ít hơn nhiều nhờ cắt tỉa.

## 7. Các cách tiếp theo

### Cách 2 — DFS đệ quy với chỉ số (theo doocs)

**Từng đoạn:**
1. `dfs(node, u)`: nếu `node is None` hoặc `node.val != arr[u]` ⇒ `False`.
2. Nếu `u == len(arr) − 1` ⇒ trả `node` là lá.
3. Ngược lại ⇒ `dfs(node.left, u + 1) or dfs(node.right, u + 1)`.

In [10]:
def valid_dfs(root, arr):
    def dfs(node, u):
        if node is None or node.val != arr[u]:
            return False
        if u == len(arr) - 1:
            return node.left is None and node.right is None
        return dfs(node.left, u + 1) or dfs(node.right, u + 1)
    return dfs(root, 0)

for arr, want in examples:
    print(arr, valid_dfs(root, arr), valid_dfs(root, arr) == want)

[0, 1, 0, 1] True True
[0, 0, 1] False True
[0, 1, 1] False True


In [11]:
class SolutionDFS:
    def isValidSequence(self, root: TreeNode, arr: list[int]) -> bool:
        def dfs(root, u):
            if root is None or root.val != arr[u]:
                return False
            if u == len(arr) - 1:
                return root.left is None and root.right is None
            return dfs(root.left, u + 1) or dfs(root.right, u + 1)

        return dfs(root, 0)

print([SolutionDFS().isValidSequence(root, a) for a, _ in examples])

[True, False, False]


**Độ phức tạp:** mỗi nút thăm tối đa một lần (độ sâu `u` cố định cho mỗi nút) và chỉ các nút có độ sâu `< len(arr)` ⇒ `O(min(N, 2^L))` ≤ `O(N)` thời gian; bộ nhớ `O(min(H, L))` cho stack đệ quy.

### Cách 3 — BFS theo tầng với tập biên

**Từng đoạn:**
1. `frontier = [root]` nếu `root.val == arr[0]`, ngược lại `[]`.
2. Với `u = 1 .. len(arr) − 1`: `frontier = [con của các nút trong frontier có val == arr[u]]`. Nếu rỗng ⇒ `False`.
3. Cuối cùng: có nút lá nào trong `frontier` ⇒ `True`.

In [12]:
def valid_bfs(root, arr):
    if root is None or root.val != arr[0]:
        return False
    frontier = [root]
    for u in range(1, len(arr)):
        frontier = [c for n in frontier for c in (n.left, n.right) if c is not None and c.val == arr[u]]
        if not frontier:
            return False
    return any(n.left is None and n.right is None for n in frontier)

for arr, want in examples:
    print(arr, valid_bfs(root, arr), valid_bfs(root, arr) == want)

[0, 1, 0, 1] True True
[0, 0, 1] False True
[0, 1, 1] False True


In [13]:
class SolutionBFS:
    def isValidSequence(self, root: TreeNode, arr: list[int]) -> bool:
        if root is None or root.val != arr[0]:
            return False
        frontier = [root]
        for u in range(1, len(arr)):
            nxt = []
            for node in frontier:
                for child in (node.left, node.right):
                    if child is not None and child.val == arr[u]:
                        nxt.append(child)
            if not nxt:
                return False
            frontier = nxt
        return any(n.left is None and n.right is None for n in frontier)

print([SolutionBFS().isValidSequence(root, a) for a, _ in examples])

[True, False, False]


**Độ phức tạp:** `O(N)` thời gian (mỗi nút xuất hiện trong tối đa một tầng), bộ nhớ `O(W)` với `W` = độ rộng lớn nhất của tập biên. Không đệ quy ⇒ an toàn với cây sâu.

### Cách 4 — DFS lặp bằng stack `(node, u)`

**Từng đoạn:**
1. `stack = [(root, 0)]`.
2. Lấy `(node, u)`; bỏ qua nếu `None` hoặc lệch.
3. Nếu `u` là chỉ số cuối: lá ⇒ `True`, không thì bỏ qua.
4. Ngược lại đẩy `(node.right, u+1)` rồi `(node.left, u+1)` (để trái được xét trước).

In [14]:
def valid_stack(root, arr):
    last = len(arr) - 1
    stack = [(root, 0)]
    while stack:
        node, u = stack.pop()
        if node is None or node.val != arr[u]:
            continue
        if u == last:
            if node.left is None and node.right is None:
                return True
            continue
        stack.append((node.right, u + 1))
        stack.append((node.left, u + 1))
    return False

for arr, want in examples:
    print(arr, valid_stack(root, arr), valid_stack(root, arr) == want)

[0, 1, 0, 1] True True
[0, 0, 1] False True
[0, 1, 1] False True


In [15]:
class SolutionStack:
    def isValidSequence(self, root: TreeNode, arr: list[int]) -> bool:
        last = len(arr) - 1
        stack = [(root, 0)]
        while stack:
            node, u = stack.pop()
            if node is None or node.val != arr[u]:
                continue
            if u == last:
                if node.left is None and node.right is None:
                    return True
                continue
            stack.append((node.right, u + 1))
            stack.append((node.left, u + 1))
        return False

print([SolutionStack().isValidSequence(root, a) for a, _ in examples])

[True, False, False]


**Độ phức tạp:** `O(N)` thời gian, `O(H)` bộ nhớ (stack chứa tối đa ~2 nút mỗi tầng). Kết hợp ưu điểm DFS (bộ nhớ theo chiều cao) và không đệ quy.

## 8. TRACE từng bước

Trace cách 2 (DFS) với `arr = [0, 1, 1, 0]` trên cây ví dụ:

| lời gọi | nút | `arr[u]` | kết quả |
|---|---|---|---|
| dfs(gốc, 0) | 0 | 0 | khớp, đi tiếp |
| dfs(L, 1) | 1 | 1 | khớp, đi tiếp |
| dfs(LL, 2) | 0 | 1 | lệch ⇒ False |
| dfs(LR, 2) | 1 | 1 | khớp, đi tiếp |
| dfs(LRL, 3) | 0 | 0 | khớp, u cuối, là lá ⇒ **True** |

In [16]:
def trace_dfs(root, arr):
    def dfs(node, u, depth, label):
        pad = "  " * depth
        if node is None:
            print(f"{pad}dfs({label}, u={u}): None -> False"); return False
        if node.val != arr[u]:
            print(f"{pad}dfs({label}, u={u}): val={node.val} != arr[{u}]={arr[u]} -> False"); return False
        if u == len(arr) - 1:
            leaf = node.left is None and node.right is None
            print(f"{pad}dfs({label}, u={u}): val={node.val} khớp, phần tử cuối, lá? {leaf} -> {leaf}")
            return leaf
        print(f"{pad}dfs({label}, u={u}): val={node.val} khớp, đi xuống")
        return dfs(node.left, u + 1, depth + 1, label + "L") or dfs(node.right, u + 1, depth + 1, label + "R")
    r = dfs(root, 0, 0, "gốc")
    print("=> kết quả:", r)
    return r

trace_dfs(root, [0, 1, 1, 0])

dfs(gốc, u=0): val=0 khớp, đi xuống
  dfs(gốcL, u=1): val=1 khớp, đi xuống
    dfs(gốcLL, u=2): val=0 != arr[2]=1 -> False
    dfs(gốcLR, u=2): val=1 khớp, đi xuống
      dfs(gốcLRL, u=3): val=0 khớp, phần tử cuối, lá? True -> True
=> kết quả: True


True

In [17]:
trace_dfs(root, [0, 1, 1])

dfs(gốc, u=0): val=0 khớp, đi xuống
  dfs(gốcL, u=1): val=1 khớp, đi xuống
    dfs(gốcLL, u=2): val=0 != arr[2]=1 -> False
    dfs(gốcLR, u=2): val=1 khớp, phần tử cuối, lá? False -> False
  dfs(gốcR, u=1): val=0 != arr[1]=1 -> False
=> kết quả: False


False

Trace cách 3 (BFS) — in tập biên ở mỗi tầng:

In [18]:
def trace_bfs(root, arr):
    if root is None or root.val != arr[0]:
        print("gốc lệch -> False"); return False
    frontier = [(root, "gốc")]
    print(f"u=0: arr[0]={arr[0]} biên = {[lbl for _, lbl in frontier]}")
    for u in range(1, len(arr)):
        nxt = []
        for n, lbl in frontier:
            for c, side in ((n.left, "L"), (n.right, "R")):
                if c is not None and c.val == arr[u]:
                    nxt.append((c, lbl + side))
        frontier = nxt
        print(f"u={u}: arr[{u}]={arr[u]} biên = {[lbl for _, lbl in frontier]}")
        if not frontier:
            print("biên rỗng -> False"); return False
    leaves = [lbl for n, lbl in frontier if n.left is None and n.right is None]
    print("các nút lá trong biên cuối:", leaves, "->", bool(leaves))
    return bool(leaves)

trace_bfs(root, [0, 1, 0, 1])

u=0: arr[0]=0 biên = ['gốc']
u=1: arr[1]=1 biên = ['gốcL']
u=2: arr[2]=0 biên = ['gốcLL']
u=3: arr[3]=1 biên = ['gốcLLR']
các nút lá trong biên cuối: ['gốcLLR'] -> True


True

## 9. Code nộp LeetCode

Dùng cách 2 (DFS với chỉ số) — đúng chữ ký theo doocs:

In [19]:
# Definition for a binary tree node.
# class TreeNode:
#     def __init__(self, val=0, left=None, right=None):
#         self.val = val
#         self.left = left
#         self.right = right
class Solution:
    def isValidSequence(self, root: TreeNode, arr: list[int]) -> bool:
        def dfs(node, u):
            if node is None or node.val != arr[u]:
                return False                       # hết cây hoặc lệch giá trị
            if u == len(arr) - 1:
                return node.left is None and node.right is None   # phải dừng ở lá
            return dfs(node.left, u + 1) or dfs(node.right, u + 1)

        return dfs(root, 0)

for arr, want in examples:
    assert Solution().isValidSequence(root, arr) == want
print("3 ví dụ: OK")

3 ví dụ: OK


## 10. Vì sao thuật toán đúng

**Mệnh đề:** `dfs(node, u)` trả `True` **khi và chỉ khi** tồn tại đường đi từ `node` xuống một lá sao cho dãy giá trị bằng `arr[u:]`.

Chứng minh quy nạp theo `len(arr) − u`:
- Nếu `node is None`: không có đường đi ⇒ `False` ✔. Nếu `node.val != arr[u]`: phần tử đầu đã lệch ⇒ `False` ✔.
- Nếu `u` là chỉ số cuối: `arr[u:]` chỉ còn 1 phần tử ⇒ đường đi phải là đúng một nút `node` và nút đó phải là lá ⇒ trả "`node` là lá" ✔.
- Ngược lại: đường đi hợp lệ phải đi qua `node` rồi xuống **một** con, phần còn lại khớp `arr[u+1:]` ⇒ theo giả thiết quy nạp, đó chính là `dfs(left, u+1) or dfs(right, u+1)` ✔.

Áp dụng với `(root, 0)` ⇒ đúng đề. **BFS:** bằng quy nạp, `frontier` sau bước `u` là đúng tập các nút độ sâu `u` mà đường từ gốc khớp `arr[0..u]`; đáp án là có lá trong tập cuối. **Stack:** duyệt cùng không gian trạng thái `(node, u)` như DFS đệ quy, chỉ khác thứ tự.

## 11. Phân tích Time / Space

Gọi `N` = số nút, `H` = chiều cao cây, `L = len(arr)`.

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Liệt kê mọi đường | `O(N · H)` | `O(N · H)` | sao chép path ở mỗi lá |
| 2. DFS đệ quy | `O(N)` (chỉ nút độ sâu `< L`) | `O(min(H, L))` stack | có thể `RecursionError` khi sâu > ~1000 |
| 3. BFS tập biên | `O(N)` | `O(W)` | an toàn với cây sâu |
| 4. DFS stack | `O(N)` | `O(min(H, L))` | an toàn, dừng sớm |

Thực tế cắt tỉa làm số nút thăm nhỏ hơn `N` rất nhiều khi `arr` lệch sớm.

In [20]:
def chain(vals):
    """Cây suy biến: mỗi nút chỉ có con trái."""
    root = cur = TreeNode(vals[0])
    for v in vals[1:]:
        cur.left = TreeNode(v); cur = cur.left
    return root

deep_vals = [random.Random(5).randint(0, 9) for _ in range(5000)]
deep = chain(deep_vals)
sys.setrecursionlimit(20000)
for name, f in [("dfs", valid_dfs), ("bfs", valid_bfs), ("stack", valid_stack)]:
    t = time.perf_counter(); r = f(deep, deep_vals); el = time.perf_counter() - t
    print(f"{name:5s}: cây sâu 5000, kết quả {r}, {el:.4f}s")

dfs  : cây sâu 5000, kết quả True, 0.0012s
bfs  : cây sâu 5000, kết quả True, 0.0006s
stack: cây sâu 5000, kết quả True, 0.0008s


## 12. Bộ test

1. Ba ví dụ đề.
2. Edge: cây một nút (khớp / không khớp), `arr` dài hơn mọi đường đi, `arr` là tiền tố của đường đi (không tới lá), nút một con, cây suy biến sâu 5000.
3. Đối chiếu ngẫu nhiên các cách với brute force trên cây ngẫu nhiên giá trị nhỏ (0..2 để dễ trùng), `arr` lấy từ đường đi thật / tiền tố / ngẫu nhiên.

In [21]:
impls = {
    "brute": valid_brute, "dfs": valid_dfs, "bfs": valid_bfs, "stack": valid_stack,
    "Solution": lambda r, a: Solution().isValidSequence(r, a),
    "SolBFS": lambda r, a: SolutionBFS().isValidSequence(r, a),
    "SolStack": lambda r, a: SolutionStack().isValidSequence(r, a),
}
more = [([0, 0, 0], True), ([0, 1, 1, 0], True), ([0, 1], False), ([0], False),
        ([0, 1, 0], False), ([0, 0], False), ([0, 1, 0, 1, 0], False), ([1], False)]
for name, f in impls.items():
    for arr, want in examples + more:
        assert f(root, arr) == want, (name, arr)
single = build_tree([5])
for name, f in impls.items():
    assert f(single, [5]) and not f(single, [4]) and not f(single, [5, 5])
    assert f(build_tree([1, 2]), [1, 2]) and not f(build_tree([1, 2]), [1])
    assert f(deep, deep_vals) and not f(deep, deep_vals[:-1]) or name == "brute"
print("ví dụ + edge: OK")

ví dụ + edge: OK


In [22]:
def rand_tree(rng, n, vmax):
    if n == 0:
        return None
    nodes = [TreeNode(rng.randint(0, vmax)) for _ in range(n)]
    for i in range(1, n):
        while True:
            p = nodes[rng.randrange(i)]
            side = rng.choice(("left", "right"))
            if getattr(p, side) is None:
                setattr(p, side, nodes[i]); break
    return nodes[0]

rng = random.Random(1430)
cnt_true = 0
for _ in range(500):
    r = rand_tree(rng, rng.randint(1, 15), rng.choice([1, 2]))
    paths = all_paths(r)
    kind = rng.random()
    if kind < 0.4:
        arr = rng.choice(paths)[:]
    elif kind < 0.6:
        p = rng.choice(paths); arr = p[:rng.randint(1, len(p))]
    else:
        arr = [rng.randint(0, 2) for _ in range(rng.randint(1, 6))]
    want = valid_brute(r, arr)
    cnt_true += want
    for name, f in impls.items():
        assert f(r, arr) == want, (name, to_level(r), arr)
print("random 500 cây: OK | số ca True:", cnt_true)

random 500 cây: OK | số ca True: 262


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Quên kiểm tra "kết thúc ở lá"

In [23]:
def wrong_no_leaf(root, arr):
    def dfs(node, u):
        if node is None or node.val != arr[u]:
            return False
        if u == len(arr) - 1:
            return True                              # SAI: chưa chắc là lá
        return dfs(node.left, u + 1) or dfs(node.right, u + 1)
    return dfs(root, 0)

for arr in ([0, 1, 1], [0, 1], [0]):
    print(arr, "| sai:", wrong_no_leaf(root, arr), "| đúng:", Solution().isValidSequence(root, arr))

[0, 1, 1] | sai: True | đúng: False
[0, 1] | sai: True | đúng: False
[0] | sai: True | đúng: False


**Vì sao sai:** đề yêu cầu đường đi **gốc → lá**. `[0, 1, 1]` chỉ là tiền tố của đường đi (ví dụ 3 của đề) ⇒ phải `false`.

**Sửa:** ở phần tử cuối trả `node.left is None and node.right is None`.

### Lỗi 2 — Kiểm tra lá bằng `or`

In [24]:
def wrong_leaf_or(root, arr):
    def dfs(node, u):
        if node is None or node.val != arr[u]:
            return False
        if u == len(arr) - 1:
            return node.left is None or node.right is None     # SAI: chỉ cần thiếu 1 con
        return dfs(node.left, u + 1) or dfs(node.right, u + 1)
    return dfs(root, 0)

for arr in ([0, 1, 0], [0, 0]):
    print(arr, "| sai:", wrong_leaf_or(root, arr), "| đúng:", Solution().isValidSequence(root, arr))

[0, 1, 0] | sai: True | đúng: False
[0, 0] | sai: True | đúng: False


**Vì sao sai:** nút `0` ở `gốc → 1 → 0` chỉ có con phải (`1`), nên `left is None` đúng ⇒ code sai coi là lá. Lá phải thiếu **cả hai** con.

**Sửa:** `and` thay vì `or`.

### Lỗi 3 — Đặt base case ở nút `None`: "hết `arr` khi gặp `None` là đúng"

In [25]:
def wrong_none_base(root, arr):
    def dfs(node, u):
        if u == len(arr):
            return node is None                    # SAI: None ở MỘT phía không có nghĩa là lá
        if node is None or node.val != arr[u]:
            return False
        return dfs(node.left, u + 1) or dfs(node.right, u + 1)
    return dfs(root, 0)

for arr in ([0, 0], [0, 1, 0]):
    print(arr, "| sai:", wrong_none_base(root, arr), "| đúng:", Solution().isValidSequence(root, arr))

[0, 0] | sai: True | đúng: False
[0, 1, 0] | sai: True | đúng: False


**Vì sao sai:** với `[0, 0]`, nút `gốc → 0` có con trái nhưng **không có con phải**; gọi `dfs(None, 2)` trả `True` vì `u == len(arr)` và nút là `None`. Thực ra nút đó **không phải lá**.

**Sửa:** kiểm tra lá **tại nút** khớp phần tử cuối (`u == len(arr) − 1`), không suy ra từ một con `None`.

### Lỗi 4 — Chỉ đi một nhánh (ưu tiên con trái nếu có)

In [26]:
def wrong_one_branch(root, arr):
    node = root
    if node is None or node.val != arr[0]:
        return False
    for u in range(1, len(arr)):
        nxt = node.left if node.left is not None else node.right     # SAI: bỏ qua nhánh phải
        if nxt is None or nxt.val != arr[u]:
            return False
        node = nxt
    return node.left is None and node.right is None

for arr in ([0, 0, 0], [0, 1, 1, 0]):
    print(arr, "| sai:", wrong_one_branch(root, arr), "| đúng:", Solution().isValidSequence(root, arr))

[0, 0, 0] | sai: False | đúng: True
[0, 1, 1, 0] | sai: False | đúng: True


**Vì sao sai:** khi cả hai con đều tồn tại, đường đúng có thể nằm bên phải (`0 → 0 → 0`) hoặc cả hai con cùng giá trị nhưng chỉ một bên dẫn tới lá phù hợp. Phải thử **cả hai** (hoặc chọn theo giá trị và vẫn giữ mọi ứng viên như BFS).

**Sửa:** `dfs(left) or dfs(right)`, hoặc BFS giữ toàn bộ tập biên.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Đường đi **gốc → lá** thoả điều kiện" ⇒ DFS mang trạng thái (chỉ số, tổng, chuỗi…) + **kiểm tra lá đúng chỗ** (Path Sum 112, Sum Root to Leaf Numbers 129).
- So khớp mảng/chuỗi với đường đi ⇒ chỉ số `u` = độ sâu; lệch ⇒ cắt tỉa.
- Lá = **cả hai** con `None`. Đừng đặt điều kiện thành công ở nút `None`.
- Cây có thể sâu tới vài nghìn ⇒ cân nhắc BFS / stack, hoặc `sys.setrecursionlimit`.
- Test nhanh: tiền tố đường đi (phải `false`), nút một con, `arr` dài hơn cây.

## 15. Biến thể của bài

1. **Path Sum (112)**: tổng gốc → lá bằng `target` — cùng khung DFS, kiểm tra lá ở cuối.
2. **Path Sum II (113)**: liệt kê mọi đường đi có tổng bằng `target`.
3. **Sum Root to Leaf Numbers (129)**: ghép chữ số dọc đường đi thành số.
4. **Đường đi khớp `arr` bắt đầu từ nút bất kỳ** (không nhất thiết gốc, không nhất thiết lá) ⇒ thử DFS từ mọi nút (giống 1367 Linked List in Binary Tree).
5. **Ký tự đại diện**: `arr[i] = -1` khớp mọi giá trị ⇒ đếm số đường đi khớp.

Code biến thể 4 (không cần bắt đầu ở gốc, không cần kết thúc ở lá):

In [27]:
def match_anywhere(root, arr):
    def from_here(node, u):
        if node is None or node.val != arr[u]:
            return False
        if u == len(arr) - 1:
            return True
        return from_here(node.left, u + 1) or from_here(node.right, u + 1)
    stack = [root]
    while stack:
        node = stack.pop()
        if node is None:
            continue
        if from_here(node, 0):
            return True
        stack.extend((node.left, node.right))
    return False

print(match_anywhere(root, [1, 1, 0]), match_anywhere(root, [0, 1, 0, 1]), match_anywhere(root, [1, 1, 1]))

True True False


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Liệt kê đường đi | Sinh mọi đường gốc → lá rồi so | `O(N·H)` | `O(N·H)` | Kiểm chứng |
| 2. DFS đệ quy + chỉ số | Lệch ⇒ dừng; cuối ⇒ kiểm tra lá | `O(N)` | `O(H)` | Mặc định, ngắn nhất (doocs) |
| 3. BFS tập biên | Tầng `u` giữ các nút khớp `arr[u]` | `O(N)` | `O(W)` | Cây sâu, tư duy theo tầng |
| 4. DFS stack | Như 2 nhưng không đệ quy | `O(N)` | `O(H)` | Cây sâu, vẫn dừng sớm |

## 17. Bài tập vận dụng

### Bài 1 (đoán output)
Trên cây ví dụ, `isValidSequence(root, [0, 0, 0])` và `isValidSequence(root, [0, 1, 0])` trả gì?

### Bài 2 (tính tay)
Chạy BFS tập biên tay với `arr = [0, 1, 1, 0]`: ghi tập biên (theo nhãn đường đi `gốc`, `L`, `R`, `LR`, …) ở mỗi tầng.

### Bài 3 (điền TODO)
Hoàn thiện `is_prefix_path(root, arr)`: `True` nếu `arr` khớp với **phần đầu** của một đường đi từ gốc (không cần tới lá).

### Bài 4 (tự viết lại)
Viết lại cách 3 bằng `deque` với phần tử `(node, u)` (BFS từng nút, không gom tầng): `valid_bfs_queue(root, arr)`.

### Bài 5 (mini challenge)
Viết `count_matching_paths(root, pattern)`: đếm số đường đi gốc → lá có độ dài đúng `len(pattern)` và khớp `pattern`, trong đó `pattern[i] == -1` khớp mọi giá trị.

In [28]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def is_prefix_path(root, arr):
    def dfs(node, u):
        # TODO: None hoặc lệch -> False; u là chỉ số cuối -> True; còn lại đi tiếp 2 nhánh
        pass
    return dfs(root, 0)

# Bài 4 — TODO
def valid_bfs_queue(root, arr):
    # TODO: q = deque([(root, 0)]); lấy ra, bỏ qua None/lệch; u cuối thì kiểm tra lá
    pass

# Bài 5 — TODO
def count_matching_paths(root, pattern):
    # TODO: DFS trả về số đường; khớp khi pattern[u] == -1 hoặc == node.val
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) `[0,0,0]`: gốc 0 → con phải 0 → con trái 0 là lá ⇒ `true`. (2) `[0,1,0]`: gốc → 1 → 0 nhưng nút 0 đó có con phải ⇒ không phải lá. (3) `true`, `false`.

**Bài 2.** (1) `u=0`: `{gốc}`. (2) `u=1` (cần 1): `{L}`; `u=2` (cần 1): `{LR}`. (3) `u=3` (cần 0): `{LRL, LRR}` — cả hai đều là lá ⇒ `true`.

**Bài 3.** (1) Giống DFS chính. (2) Khác duy nhất: ở phần tử cuối trả `True` (không kiểm tra lá). (3) `[0,1,1]` ⇒ `True`, `[0,0,1]` ⇒ `False`.

**Bài 4.** (1) `while q: node, u = q.popleft()`. (2) Bỏ qua nếu `None` hoặc `node.val != arr[u]`. (3) `u == len(arr)-1`: lá ⇒ `True`; ngược lại đẩy hai con với `u + 1`.

**Bài 5.** (1) `ok = pattern[u] == -1 or pattern[u] == node.val`. (2) Ở `u` cuối: trả `1` nếu lá, `0` nếu không. (3) Ngược lại cộng kết quả hai con (không dùng `or` vì cần đếm hết).

## 19. Đáp án tham khảo

In [29]:
# Bài 1
assert Solution().isValidSequence(root, [0, 0, 0]) is True
assert Solution().isValidSequence(root, [0, 1, 0]) is False

# Bài 2 (kiểm tra bằng cách lấy nhãn tập biên)
def frontier_labels(root, arr):
    if root.val != arr[0]:
        return []
    fr, out = [(root, "gốc")], [["gốc"]]
    for u in range(1, len(arr)):
        fr = [(c, l + s) for n, l in fr for c, s in ((n.left, "L"), (n.right, "R")) if c and c.val == arr[u]]
        out.append([l for _, l in fr])
    return out
assert frontier_labels(root, [0, 1, 1, 0]) == [["gốc"], ["gốcL"], ["gốcLR"], ["gốcLRL", "gốcLRR"]]

# Bài 3
def is_prefix_path_sol(root, arr):
    def dfs(node, u):
        if node is None or node.val != arr[u]:
            return False
        if u == len(arr) - 1:
            return True
        return dfs(node.left, u + 1) or dfs(node.right, u + 1)
    return dfs(root, 0)

# Bài 4
def valid_bfs_queue_sol(root, arr):
    q = deque([(root, 0)])
    last = len(arr) - 1
    while q:
        node, u = q.popleft()
        if node is None or node.val != arr[u]:
            continue
        if u == last:
            if node.left is None and node.right is None:
                return True
            continue
        q.append((node.left, u + 1))
        q.append((node.right, u + 1))
    return False

# Bài 5
def count_matching_paths_sol(root, pattern):
    def dfs(node, u):
        if node is None or (pattern[u] != -1 and pattern[u] != node.val):
            return 0
        if u == len(pattern) - 1:
            return 1 if node.left is None and node.right is None else 0
        return dfs(node.left, u + 1) + dfs(node.right, u + 1)
    return dfs(root, 0)

assert count_matching_paths_sol(root, [0, 1, 1, 0]) == 2
assert count_matching_paths_sol(root, [-1, -1, -1, -1]) == 3
assert count_matching_paths_sol(root, [-1, -1, -1]) == 1

rng = random.Random(77)
for _ in range(400):
    r = rand_tree(rng, rng.randint(1, 12), 1)
    paths = all_paths(r)
    arr = [rng.randint(0, 1) for _ in range(rng.randint(1, 5))]
    pref = any(p[:len(arr)] == arr for p in paths)
    assert is_prefix_path_sol(r, arr) == pref
    assert valid_bfs_queue_sol(r, arr) == valid_brute(r, arr)
    pat = [rng.choice([-1, 0, 1]) for _ in range(rng.randint(1, 5))]
    exp = sum(1 for p in paths if len(p) == len(pat) and all(a == -1 or a == b for a, b in zip(pat, p)))
    assert count_matching_paths_sol(r, pat) == exp
print("Tất cả đáp án bài tập: OK")

Tất cả đáp án bài tập: OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template DFS "đường gốc → lá khớp dãy":**

```python
def dfs(node, u):
    if node is None or node.val != arr[u]:
        return False
    if u == len(arr) - 1:
        return node.left is None and node.right is None
    return dfs(node.left, u + 1) or dfs(node.right, u + 1)
return dfs(root, 0)
```

**Template BFS tập biên:**

```python
fr = [root] if root and root.val == arr[0] else []
for u in range(1, len(arr)):
    fr = [c for n in fr for c in (n.left, n.right) if c and c.val == arr[u]]
return any(n.left is None and n.right is None for n in fr)
```

**Checklist:**
- [ ] Kiểm tra lá ở phần tử **cuối** của `arr`.
- [ ] Lá = `left is None **and** right is None`.
- [ ] Không coi "con `None`" là điểm kết thúc hợp lệ.
- [ ] Thử **cả hai** nhánh.
- [ ] Không truy cập `arr[u]` khi `u >= len(arr)`.
- [ ] Cây sâu 5000: dùng BFS/stack hoặc tăng giới hạn đệ quy.
- [ ] Đã test: tiền tố đường đi, nút một con, `arr` dài hơn cây.